In [1]:
from config import init_env
from config import variables
import importlib
variables = importlib.reload(variables)
init_env.set_environment_variables()
import requests


### Hana Database

#### Create the connection to Hana database

In [2]:
from hdbcli import dbapi
# Initial the cursor
connection=dbapi.connect(
        address="e41c3eb7-55e9-47db-8915-e1ab64b9872a.hna0.prod-eu10.hanacloud.ondemand.com",
        port="443",
        user="USR_7K4HHI6O79L4LB691X7CN6MUQ",
        password="Jx8Q6g6l7NSZIMnooxdlCdkCLdEiR9--w4NSU4A0uZKfvLbrjKkgCJ9V2BXRHkQKP9ENitstBv4nv8.OraOa27HlVGwl50.D9BoaNPhoUDl-oGEbek1.n7QmYm-i3r.d",
        autocommit=True,
        sslValidateCertificate=False
    )

In [3]:
# Initial the cursor
cursor = connection.cursor() 

#### Create a HANA database table

In [26]:
# Create a custom table with attribute

table_name = "LANGCHAIN_DEMO_SELF_QUERY"
try:
  cursor.execute(
      f'''
      CREATE TABLE "{table_name}" (
        "id"        INTEGER PRIMARY KEY,
        "name"      NVARCHAR(100),
        "is_active" BOOLEAN,
        "height"    DOUBLE,
        "VEC_TEXT"  NCLOB,
        "VEC_META"  NCLOB,
        "VEC_VECTOR" REAL_VECTOR(768)
      )
      '''
  )
  print(f'Table "{table_name}" created in the SAP HANA database.')

except Exception:
    print(f"Table  {table_name} already exsits.")
    pass

Table  LANGCHAIN_DEMO_SELF_QUERY already exsits.


![](./images/TableCreation.png)

#### (Optional) Delete HANA database table

In [23]:
# Delete exsiting table if exists
try:
    cursor.execute('DROP TABLE "LANGCHAIN_DEMO_SELF_QUERY"')
    print("Table dropped successfully.")
except Exception:
    print("No existing table.")
    pass

No existing table.


In [14]:

try:
    cur.execute('DROP TABLE "LANGCHAIN_DEMO_SELF_QUERY"')
    print("Table dropped successfully.")
except Exception:
    print("No existing table.")
    pass


No existing table.


In [27]:
# Step 1: Load documents

from langchain_community.document_loaders import PyPDFDirectoryLoader
DATA_PATH = r"datafiles"
loader = PyPDFDirectoryLoader(DATA_PATH)
documents = loader.load()
print(f"Loaded {len(documents)} documents.")

Loaded 6 documents.


In [30]:
# Step 2: Chunk documents

from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=150,
    length_function=len,
)
split_documents = text_splitter.split_documents(documents)
print(f"Split into {len(split_documents)} chunks.")

Split into 25 chunks.


In [32]:
# Step 3: Set embedding model  
 
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI, OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(deployment_id=variables.EMBEDDING_DEPLOYMENT_ID)  # Deployment ID of text-embedding-3-large

In [39]:
from langchain_hana import HanaDB

table_name="TEST_EMBEDDING_TABLE"
 

db = HanaDB(
    embedding=embedding_model, 
    connection=connection, 
    table_name=table_name
)

db.add_documents(split_documents)
print(f"Table {db.table_name} created in the SAP HANA database.")

Table TEST_EMBEDDING_TABLE created in the SAP HANA database.


<i><b>db.add</b></i> will create the table using the given table name if it does not exsit.
The table is defaulted with 3 fields
![](./images/db_add.png)</p>
If the table already exsits, <i><b>db.add</b></i> will add new entries into the table.
![](./images/DuplicateEntries.png)</p>

In [37]:
# Need to delete this table for repeating test
try:
    cursor.execute(f'DROP TABLE "{table_name}"')
    print(f'Table "{table_name}" dropped successfully.')
except Exception:
    print(f'Table "{table_name}" does not exist.')


Table "TEST_EMBEDDING_TABLE" does not exist.


In [52]:
from IPython.display import Markdown
 

# Use `db.table_name` instead of `variables.EMBEDDING_TABLE` because HANA driver sanitizes a table name by removing unaccepted characters
is_ok = cur.execute(f'''SELECT "VEC_TEXT", "VEC_META", TO_NVARCHAR("VEC_VECTOR") FROM "{db.table_name}"''')
record_columns=cur.fetchone()
if record_columns:
    display({"VEC_TEXT" : record_columns[0], "VEC_META" : eval(record_columns[1]), "VEC_VECTOR" : record_columns[2]})


{'VEC_TEXT': 'Introduction \nWe SAP are excited to announce that we have started working on a VS Code extension for \nABAP . We understand that the community has high expectations, and we want to \ncommunicate transparently about what you can expect from ABAP Development Tools for \nVS Code. In this article, we will share details about the scope of the ﬁrst release and what’s \nplanned for the future. \nSee related article: Behind the Design: How We Transformed the ABAP Development Tools',
 'VEC_META': {'producer': 'Microsoft: Print To PDF',
  'creator': 'PyPDF',
  'creationdate': '2025-11-18T10:40:21+08:00',
  'author': 'SUN Yufeng (BD/PTD-SPR1)',
  'moddate': '2025-11-18T10:40:21+08:00',
  'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx',
  'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf',
  'total_pages': 2,
  'page': 0,
  'page_label': '1'},
 'VEC_VECTOR': '[0.011229125,-0.02020525,-0.009772568,

In [53]:
connection.isconnected()

True

In [54]:
from gen_ai_hub.proxy.langchain.init_models import init_llm
model = init_llm(
    'gpt-4o', 
    temperature=0.1,
    max_tokens=8000
)


In [56]:
model = ChatOpenAI(deployment_id=variables.LLM_DEPLOYMENT_ID)  # LLM deployment ID. Here gpt-4o has been maintained

In [62]:
retriever = db.as_retriever(search_kwargs={"k": 1})

In [75]:

from langchain.chains import RetrievalQA
# Create the QA instance to query llm based on custom documents
qa = RetrievalQA.from_llm(llm=model, retriever=retriever, return_source_documents=True)

# Send query
query = "Why is ABAP in VS Code is so appealing?"

answer = qa.invoke(query)
display(answer["result"])


'The appeal of ABAP in Visual Studio Code (VS Code) is primarily driven by the demand from users for more flexibility in choosing their development environment. VS Code is a popular IDE that offers a streamlined and customizable user experience, which can enhance productivity and make development more efficient. Users have specifically requested support for ABAP in VS Code, indicating a preference for its lightweight design, rich extension ecosystem, and cross-platform capabilities. These features make it an attractive option for developers looking for versatility and modern tooling in their workflow.'

In [66]:


for document in answer['source_documents']:
    display(document.metadata)   
    print(document.page_content)



{'producer': 'Microsoft: Print To PDF',
 'creator': 'PyPDF',
 'creationdate': '2025-11-18T10:42:07+08:00',
 'author': 'SUN Yufeng (BD/PTD-SPR1)',
 'moddate': '2025-11-18T10:42:07+08:00',
 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx',
 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf',
 'total_pages': 4,
 'page': 0,
 'page_label': '1'}

Introduction 
Currently, oƯicial ABAP tool support exists for SAP GUI and Eclipse. For years, users 
have asked us to bring this support to additional IDEs. Based on the user survey 
results from 2023 and 2025, the most requested development environment is Visual 
Studio Code (VS Code). However, many users have also expressed interest in other 
environments such as JetBrains IDEs, Neovim, or even Zed. In short, our user base 
wants more ﬂexibility when choosing their development environment.
